In [1]:
import os
from pathlib import Path

project_root = Path.cwd().parent
os.chdir(project_root)
print("CWD:", project_root)

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from src.rag_pipeline import RagPipeline

CWD: /home/gusevsaint/Workspace/study/practice/mental-helper


/home/gusevsaint/Workspace/study/practice/mental-helper/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
rag = RagPipeline()
gen_device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("google-t5/t5-base")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-base").to(gen_device)

In [ ]:
def generate_answer(query, top_k=4, max_new_tokens=220):
    retrieved = rag.retrieve(query, top_k=top_k)
    context = "\n\n".join(
        f"[{i}] Q: {r['question']}\n[{i}] A: {r['answer']}"
        for i, r in enumerate(retrieved, 1)
    )
    context = context[:4000]

    prompt = f"""You are a mental health assistant. Provide 4-6 clear, actionable points.
Avoid generic advice. End with an encouraging statement.

Question: {query}

Context:
{context}

Answer:"""

    inputs = tok(prompt, return_tensors="pt", truncation=True).to(gen_device)
    outputs = gen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05,
        no_repeat_ngram_size=3,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
    )
    answer = tok.decode(outputs[0], skip_special_tokens=True)
    return {"answer": answer, "retrieved": retrieved}


def generate_baseline(query, max_new_tokens=220):
    prompt = f"""You are a mental health assistant. Provide 4-6 clear, actionable points.
Avoid generic advice. End with an encouraging statement.

Question: {query}

Answer:"""

    inputs = tok(prompt, return_tensors="pt", truncation=True).to(gen_device)
    outputs = gen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05,
        no_repeat_ngram_size=3,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
    )
    return tok.decode(outputs[0], skip_special_tokens=True)


# comparison
resp_rag = generate_answer("How to cope with stress?", top_k=4)
resp_base = generate_baseline("How to cope with stress?")

print("With RAG:\n", resp_rag["answer"], "\n")
print("Sources:")
for r in resp_rag["retrieved"]:
    print("-", r["question"])

print("\nWithout RAG:\n", resp_base)

With RAG:
 орос: олуат и доров.  инес авери и клие. . как еа телеон авротив и свои у то . 

Sources:
- How to manage stress?
- Are There Coping Factors To Help Deal Effectively With Stress?
- How to cope up with social isolation?
- I've tried various coping strategies, but nothing seems to be working. Can you help me identify the most appropriate coping mechanisms for my specific situation?

Without RAG:
 омоник  менталному доров. а 4–6 тки унктов с конкретнми дествими. - то еавери тлм онадиваием редлоением. – наителн


In [8]:
# тест
resp = generate_answer("How to manage stress?", top_k=3)
print("Ответ:\n", resp["answer"])
print("\nИсточники (Q/A):")
for r in resp["retrieved"]:
    print("-", r["question"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Ответ:
 орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: орос: о

Источники (Q/A):
- How to manage stress?
- Are There Coping Factors To Help Deal Effectively With Stress?
- I'm dealing with financial hardships that contribute to my stress and anxiety. How can I access resources and support to manage these challenges?
